# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**LUIS JOSE PAREDES RAMIREZ**

El siguiente ejercicio de busqueda se implemento Búsqueda A\* usando la heurística dada en el aula virtual

$$f(n) = c(n) + h(n)$$


Para la funcion de costo $c(n)$ se utilizo el atributo `accumulated_cost` propio de cada estado y para la funcion heuristica utilizamos como regla principal $h(n)$ bonifica con un punto negativo por cada disco ubicado en la posición correcta.

---


Condiciones Adicionales:

- La función debe devolver la salida correspondiente a la solución encontrada o `None` si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [ ]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi
from aima_libs.aima import PriorityQueue as AimaPriorityQueue

def c(node):
    if node.parent:
        return node.state.accumulated_cost - node.parent.state.accumulated_cost
    return node.state.accumulated_cost

def h(node, number_disks):
    disks_on_peg3 = node.state.get_state_dict().get('peg_3')
    count = 0

    # Los discos deben estar siempre en orden [5,4,3,2,1]
    for disk in range(number_disks, 0, -1):
        if disk in disks_on_peg3:
            count = count + 1
        else:
            break
    return -1*count

def f(node, number_disks):
    return c(node) + h(node, number_disks)

def search_algorithm(number_disks=5) -> (NodeHanoi, dict):
    
    # Inicializamos el problema
    list_disks = [i for i in range(5, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    # Cola prioritaria
    frontier = AimaPriorityQueue(order='min', f=lambda node: f(node, number_disks))
    frontier.append(NodeHanoi(problem.initial))

    # Conjunto de estados ya visitados
    explored = set()

    max_depth = 0
    node_explored = 0
    
    while len(frontier) != 0:
        _priority_value, node = frontier.pop()
        node_explored += 1

        if max_depth < node.depth:
            max_depth = node.depth
        
        explored.add(node.state) 
        
        # Verificamos si llegamos al objetivo
        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": node_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": max_depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics
        
        # Agregamos a la frontera los nodos sucesores que no hayan sido visitados
        for next_node in node.expand(problem):
            if next_node.state not in explored:
                frontier.append(next_node)

    # Si no se encuentra solución, devolvemos métricas igualmente
    metrics = {
        "solution_found": False,
        "nodes_explored": node_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth, 
        "cost_total": None,
    }
    return None, metrics

Se prueba la función:

In [26]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [27]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 160
states_visited: 110
nodes_in_frontier: 46
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [28]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [29]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3


# Simulacion

In [30]:
solution.generate_solution_for_simulator()

Mover los archivos `initial_state.json` & `sequence.json` dentro de la carpeta `simulator/` y ejecutar

```
python3 ./simulation_hanoi.py
```